In [1]:
from osgeo import gdal
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from tqdm import tqdm
import subprocess
import os

In [9]:
def extract_info(dataset, footprints, crs,year, filename):   
    # Get the image info

    ###
    def extract_houses(image_bbox): 
        # Get the house info within an image
        
        regions_gdf = gpd.GeoDataFrame({'geometry': [image_bbox]}, crs=crs) 
        # Transform regions to WGS 84 to match city boundary
        regions_gdf = regions_gdf.to_crs('EPSG:3857')
        image = regions_gdf.geometry.unary_union
        id = []
        footprints_in_image = footprints[footprints.intersects(image)]
        
        for index, footprint in footprints_in_image.iterrows():
            footprint_in_image_bbox = footprint.geometry.intersection(image)
        
            # Calculate the areas
            original_area = footprint.geometry.area
            intersection_area = footprint_in_image_bbox.area
            
            # Calculate the percentage of the footprint area within the image bounding box
            percentage_in_image = (intersection_area / original_area) * 100
            
            # Only proceed if the percentage is greater than 70%
            if percentage_in_image <= 70:
                continue
            id.append(str(int(footprint['OBJECTID'])))
            
        return ",".join(id)
    ###
    driver = "{}/{}".format(dataset.GetDriver().ShortName,
                                dataset.GetDriver().LongName)
    
    size = "{},{},{}".format(dataset.RasterXSize,
                                        dataset.RasterYSize,
                                        dataset.RasterCount)
    
    # projection = dataset.GetProjection()
    geotransform = dataset.GetGeoTransform()
    
    if geotransform:
        pixel = "({},{})".format(geotransform[1], geotransform[5])
        minx = geotransform[0]
        maxy = geotransform[3]
        maxx = minx + (geotransform[1] * dataset.RasterXSize)
        miny = maxy + (geotransform[5] * dataset.RasterYSize)
        
        # Create a bounding box for the satellite image
        image_bbox = box(minx, miny, maxx, maxy)  

        # Get the id of houses in this image
        id = extract_houses(image_bbox)
    else:
        origin = 0
        pixel = 0
        image_bbox = 0
        print(f"Cant get the geoinfo of the picture {filename}")

    return [year, filename, driver, size, crs, pixel, image_bbox, id]

def list_geo_files(folder_path):
    # List all files in the given directory
    all_files = os.listdir(folder_path)
    
    # Filter to include only .tif files
    tiff_files = [file for file in all_files if (file.endswith('.tif') or file.endswith('.tiff')) and (not file.startswith('._'))]
    
    return tiff_files

# Get the image info

In [2]:
footprints = gpd.read_file('C:/Users/Lenovo/Desktop/python/ra/Solar-Panel/testing/boundaries/edited boundary/PROPERTY_BOUNDARIES_WGS84_modified.shp')

In [11]:
year = 2022
crs = 'EPSG:2952'

# Folder of images
image_path = f'F:/UofT_CoT_Orthos_Request/CityOfTorontoOrthos_{year}_TorontoCD/'
pictures = list_geo_files(image_path)
infos = []

for p in tqdm(pictures):
    dataset = gdal.Open(image_path + p, gdal.GA_ReadOnly)
    info = extract_info(dataset, footprints, crs,  year, p)
    infos.append(info)
    # Close the dataset
    dataset = None

df = pd.DataFrame(infos, columns=['Time', 'Name', 'Driver', 'Size', 'Crs', 'Pixel', 'Boundary', 'House ids'])

#Foler to save the data
output_path = f'C:/Users/Lenovo/Desktop/python/ra/Solar-Panel/data/images_check/{year}_image_check_test.xlsx'
df.to_excel(output_path, index=False)

100%|██████████████████████████████████████████████████████████████████████████████| 2751/2751 [07:01<00:00,  6.52it/s]
